## Adding DACCS to the PyPSA-Eur Belgium Tutorial

The following notebook is designed as a proof of concept for integrating DACCS (Direct Air Carbon Capture and Storage) into a simple PyPSA-Eur electricity system example and analyzing emissions reductions with DACCS.

This notebook goes through the following:

1. Establishing a baseline electricity system in Belgium via the PyPSA-Eur tutorial: https://pypsa-eur.readthedocs.io/en/latest/tutorial/

2. Calculating baseline operational CO2 emissions.

3. Lowering the carbon emissions cap and recalculating the baseline system. 

4. Integrating DACCS as a load into the network.

5. Compare the CO2 emissions before and after DACCS int

## Scope

Following and adapting the tutorial located here: https://pypsa-eur.readthedocs.io/en/latest/tutorial/

Region: Belgium |
Time period: March 1st through 8th, 2013 |
Initial system: electricity only (no sector coupling)

This exploratory exercise is designed to introduce DACCS into a small regional electricity network under a tight carbon cap.

This model does not attempt to represent real-world current or projected future costs of DACCS. 

Additionally, DACCS is added purely as a load to the system and carbon capture is not considered for satisfying the carbon emissions constraint. 

## References

Note: ChatGPT was used to help guide me through this exercise. Additional references I used for understanding the EU ETS, potential CDR integration into the EU ETS, DACCS, BECCS, and ways to incorporate DACCS into this model include:

1. Sultani, D., Osorio, S., Günther, C., Pahle, M., Sievert, K., Schmidt, T. S., Steffen, B., & Edenhofer, O. (2026). How the EU can utilize its carbon market to scale up carbon dioxide removal. *Joule*, 10(7), 102395. [https://doi.org/10.1016/j.joule.2026.102395](https://doi.org/10.1016/j.joule.2026.102395)

2. European Commission, Directorate-General for Climate Action. (2025). *EU Emissions Trading System (EU ETS)*. Climate Action. [https://climate.ec.europa.eu/areas-action/carbon-markets/eu-emissions-trading-system-eu-ets_en](https://climate.ec.europa.eu/areas-action/carbon-markets/eu-emissions-trading-system-eu-ets_en) (accessed 3 November 2026)

3. Carbon Gap. (2026). *EU Emissions Trading System (EU ETS)*. Carbon Gap Policy Tracker. [https://tracker.carbongap.org/policy/eu-emissions-trading-system/](https://tracker.carbongap.org/policy/eu-emissions-trading-system/) (accessed 3 November 2026)

4. Sultani, D., & Leisinger, C. (2026). *Let's be radically honest about the EU ETS*. Potsdam Institute for Climate Impact Research. [https://www.pik-potsdam.de/members/sultani/sultanileisinger2026_radicalhonestyeuets.pdf](https://www.pik-potsdam.de/members/sultani/sultanileisinger2026_radicalhonestyeuets.pdf) (AI-assisted translation of the German original, "Zeit für radikale Ehrlichkeit," Bertelsmann Stiftung.)

In [1]:
import pypsa
import pandas as pd
import matplotlib.pyplot as plt

network_path = "../results/test-elec/networks/solved_2050.nc"

n = pypsa.Network(network_path)

# rename the network

n.name = "Belgium No DACCS"

print(n)

INFO:pypsa.network.io:New version 1.4.0 available! (Current: 1.2.4)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks


PyPSA Network 'Belgium No DACCS'


In [2]:
# calculate electricity generation and account for snapshot weights

energy = n.generators_t.p.mul(
    n.snapshot_weightings.generators, axis = 0
).sum()

# convert electricity generation into fuel consumption

fuel = energy / n.generators.efficiency

# calculate emissions for each generator

emissions = fuel * n.generators.carrier.map(
    n.carriers.co2_emissions
)

# adding up emissions by technology
baseline_emissions_by_technology = (
    emissions.groupby(n.generators.carrier).sum()
)

# calculate total baseline emissions
baseline_emissions = emissions.sum()

print("Baseline emissions by technology:")
display(baseline_emissions_by_technology)

print(f"\nTotal baseline emissions: {baseline_emissions:,.2f} tonnes CO2")

Baseline emissions by technology:


carrier
CCGT             81022.113887
biomass              0.000000
nuclear              0.000000
offwind-ac           0.000000
offwind-dc           0.000000
offwind-float        0.000000
oil                  0.096940
onwind               0.000000
solar                0.000000
solar-hsat           0.000000
waste            41866.953906
dtype: float64


Total baseline emissions: 122,889.16 tonnes CO2


In [3]:
print(baseline_emissions)

122889.16473324648


In [4]:
# take a look at avg costs per mwh for each generator
avg_costs_by_technology =   n.generators.groupby("carrier")["marginal_cost"].mean()
print(avg_costs_by_technology)

carrier
CCGT              43.282657
biomass           19.990624
nuclear           27.332927
offwind-ac         0.024867
offwind-dc         0.024862
offwind-float      0.025476
oil              124.777099
onwind             0.025292
solar              0.019643
solar-hsat         0.019822
waste             34.123506
Name: marginal_cost, dtype: float64


In [5]:
# as well as the total cost of the system
total_system_cost = n.statistics.system_cost()
print(total_system_cost)

component    carrier                 
Generator    Combined-Cycle Gas          2.671287e+07
             Offshore Wind (AC)          1.459015e+07
             Offshore Wind (DC)          8.882720e+00
             Offshore Wind (Floating)    1.116215e+01
             Onshore Wind                3.929956e+06
             Solar                       2.751293e+07
             biomass                     1.184300e+06
             nuclear                     8.539349e+07
             oil                         1.231900e+05
             solar-hsat                  1.314087e+02
             waste                       6.987859e+06
Line         AC                          1.535426e+06
Link         H2 Electrolysis             4.596946e+01
             H2 Fuel Cell                2.216048e+01
StorageUnit  Battery Storage             1.400196e+02
Store        Hydrogen Storage            4.137143e+01
dtype: float64


In [6]:
# get the global co2 constraint for this system
print(n.global_constraints)

                          type  investment_period bus carrier_attribute sense  \
name                                                                            
CO2Limit-upper  primary_energy                NaN         co2_emissions    <=   

                    constant        mu  
name                                    
CO2Limit-upper  1.917808e+06 -0.000018  


In [7]:
# note here that the CO2 is much higher than the baseline emissions
print(baseline_emissions)

122889.16473324648


Since the initial baseline model has a high CO2 cap, we'll lower the cap so the system has to actually make some decisions about when not to emit. 

In [8]:
n_tight_cap = pypsa.Network(network_path)

# make the new cap tighter than baseline emissions
new_cap = baseline_emissions * 0.90

n_tight_cap.global_constraints.loc[
    "CO2Limit-upper", "constant"
] = new_cap

print("Updated network cap:")

print(
    n_tight_cap.global_constraints.loc[
        "CO2Limit-upper", "constant"
    ]
)

INFO:pypsa.network.io:New version 1.4.0 available! (Current: 1.2.4)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks


Updated network cap:
110600.24825992183


In [9]:
# reoptimize the network

status, condition = n_tight_cap.optimize(
    solver_name="highs"
)

print("Status:", status)
print("Condition:", condition)

/tmp/ipykernel_23522/1301998315.py:3: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  status, condition = n_tight_cap.optimize(
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.36s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 570 primals, 1203 duals
Objective: 5.01e+07
Solver: highs
Runtime: 0.01s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



Running HiGHS 1.13.1 (git hash: n/a): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-d3qb9_si has 1196 rows; 570 cols; 2440 nonzeros
Coefficient ranges:
  Matrix  [3e-02, 3e+01]
  Cost    [2e-01, 2e+04]
  Bound   [1e+08, 1e+08]
  RHS     [2e+01, 1e+05]
Presolving model
609 rows, 560 cols, 1824 nonzeros  0s
Dependent equations search running on 105 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
567 rows, 518 cols, 1789 nonzeros  0s
Presolve reductions: rows 567(-629); columns 518(-52); nonzeros 1789(-651) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -9.0857393090e-02 Ph1: 139(216527); Du: 9(0.0908574) 0.0s
        384     5.0132755753e+07 Pr: 0(0); Du: 0(4.78308e-13) 0.0s
        384     5.0132755753e+07 Pr: 0(0); Du: 0(4.78308e-13) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Mod

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Line-ext-s-lower, Line-ext-s-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, Kirchhoff-Voltage-Law, StorageUnit-energy_balance, Store-energy_balance were not assigned to the network.


Status: ok
Condition: optimal


In [10]:
# perform emissions calculations on tighter cap network
tight_energy = n_tight_cap.generators_t.p.mul(
    n_tight_cap.snapshot_weightings.generators,
    axis=0
).sum()

tight_fuel = (
    tight_energy / n_tight_cap.generators.efficiency
)

tight_emissions_by_generator = (
    tight_fuel
    * n_tight_cap.generators.carrier.map(
        n_tight_cap.carriers.co2_emissions
    )
)

tight_emissions = tight_emissions_by_generator.sum()

print(f"New emissions: {tight_emissions:,.2f} tonnes CO2")
print(f"CO2 cap: {new_cap:,.2f} tonnes CO2")

New emissions: 110,600.25 tonnes CO2
CO2 cap: 110,600.25 tonnes CO2


## Adding DACCS

Here, I'll first test adding DACCS to the network as a load. Carbon capture will not go into the optimization of the model (carbon storage is not implemented). 

In [11]:
# creating a new network with DACCS
n_daccs = pypsa.Network(network_path)

# add the tight cap to this network as well
n_daccs.global_constraints.loc[
    "CO2Limit-upper", "constant"
    ] = new_cap
n_daccs.name = "Belgium with DACCS"

# input the fake DACCS data
daccs_mw = 10
daccs_electricity_per_co2 = 1.5

# add DACCS electricity demand at the BE0 4AC bus 
# (random choice here, but we would want to consider operational requirements later)
n_daccs.add(
    "Load",
    "DACCS electricity demand",
    bus="BE0 4AC",
    p_set=daccs_mw
)

# calc total electricity consumption
daccs_electricity = (
    daccs_mw * n_daccs.snapshot_weightings.generators.sum()
)

#calc amount of CO2 captured
daccs_captured = (
    daccs_electricity / daccs_electricity_per_co2
)

print(f"DACCS electricity consumption: {daccs_electricity:,.0f} MWh")
print(f"CO2 captured: {daccs_captured:,.0f} tonnes")

INFO:pypsa.network.io:New version 1.4.0 available! (Current: 1.2.4)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks


DACCS electricity consumption: 1,680 MWh
CO2 captured: 1,120 tonnes


In [12]:
print(
    n_daccs.global_constraints.loc[
        "CO2Limit-upper",  ["constant", "mu"]
    ]
)

constant    110600.248260
mu              -0.000018
Name: CO2Limit-upper, dtype: float64


In [13]:
n_daccs.optimize(solver_name="highs")

/tmp/ipykernel_23522/3948332610.py:1: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n_daccs.optimize(solver_name="highs")
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.34s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 570 primals, 1203 duals
Objective: 5.02e+07
Solver: highs
Runtime: 0.01s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



Running HiGHS 1.13.1 (git hash: n/a): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-w_wsdw6e has 1196 rows; 570 cols; 2440 nonzeros
Coefficient ranges:
  Matrix  [3e-02, 3e+01]
  Cost    [2e-01, 2e+04]
  Bound   [1e+08, 1e+08]
  RHS     [2e+01, 1e+05]
Presolving model
609 rows, 560 cols, 1824 nonzeros  0s
Dependent equations search running on 105 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
567 rows, 518 cols, 1789 nonzeros  0s
Presolve reductions: rows 567(-629); columns 518(-52); nonzeros 1789(-651) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -9.0857393090e-02 Ph1: 139(216527); Du: 9(0.0908574) 0.0s
        365     5.0199758611e+07 Pr: 0(0); Du: 0(6.25278e-13) 0.0s
        365     5.0199758611e+07 Pr: 0(0); Du: 0(6.25278e-13) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Mod

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Line-ext-s-lower, Line-ext-s-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, Kirchhoff-Voltage-Law, StorageUnit-energy_balance, Store-energy_balance were not assigned to the network.


('ok', 'optimal')

In [14]:
# comparing how much electricity was produced with and wihtout DACCS for tight cap system

tight_cap_baseline_generation = (
    n_tight_cap.generators_t.p
    .mul(n_tight_cap.snapshot_weightings.generators, axis=0)
    .sum()
    .sum()
)

daccs_generation = (
    n_daccs.generators_t.p
    .mul(n_daccs.snapshot_weightings.generators, axis=0)
    .sum()
    .sum()
)

print(f"Baseline with tight cap generation: {tight_cap_baseline_generation:,.2f} MWh")
print(f"DACCS scenario generation: {daccs_generation:,.2f} MWh")
print(f"Difference: {daccs_generation - tight_cap_baseline_generation:,.2f} MWh")

Baseline with tight cap generation: 1,743,999.00 MWh
DACCS scenario generation: 1,745,679.00 MWh
Difference: 1,680.00 MWh


This makes sense, because DACCS just added a demand to the system of 1,680.00 MWh.

In [15]:
# electricity generated by each generator (MWh)
daccs_energy = n_daccs.generators_t.p.mul(
    n_daccs.snapshot_weightings.generators,
    axis=0
).sum()

# converting electricity output into fuel input (MWh)
daccs_fuel = daccs_energy / n_daccs.generators.efficiency

# calc emissions (tonnes CO2)
daccs_emissions_by_generator = (
    daccs_fuel
    * n_daccs.generators.carrier.map(
        n_daccs.carriers.co2_emissions
    )
)

# emissions by technology
daccs_emissions_by_technology = (
    daccs_emissions_by_generator
    .groupby(n_daccs.generators.carrier)
    .sum()
)

daccs_emissions = daccs_emissions_by_generator.sum()

print("DACCS scenario emissions by technology:")
display(daccs_emissions_by_technology)

print(f"Total DACCS scenario emissions: {daccs_emissions:,.2f} tonnes CO2")

DACCS scenario emissions by technology:


carrier
CCGT             83416.494657
biomass              0.000000
nuclear              0.000000
offwind-ac           0.000000
offwind-dc           0.000000
offwind-float        0.000000
oil                  0.000000
onwind               0.000000
solar                0.000000
solar-hsat           0.000000
waste            27183.753603
dtype: float64

Total DACCS scenario emissions: 110,600.25 tonnes CO2


In [16]:
print(f"Baseline emissions: {tight_emissions:,.2f} tonnes CO2")
print(f"DACCS emissions: {daccs_emissions:,.2f} tonnes CO2")


Baseline emissions: 110,600.25 tonnes CO2
DACCS emissions: 110,600.25 tonnes CO2


In [17]:
# still need to take daccs carbon capture into account

print(f"Baseline emissions: {tight_emissions:,.2f} tonnes CO2")
# add in carbon capture
daccs_net_emissions = daccs_emissions - daccs_captured
print(f"DACCS net emissions: {daccs_net_emissions:,.2f} tonnes CO2")

print(
    f"Change in emissions: "
    f"{daccs_net_emissions - baseline_emissions:,.2f} tonnes CO2"
)


Baseline emissions: 110,600.25 tonnes CO2
DACCS net emissions: 109,480.25 tonnes CO2
Change in emissions: -13,408.92 tonnes CO2


## Summary and Limitations

The original PyPSA-Eur Belgium system had a CO2 cap of ~1.92 Mt over the 7-day window, far above baseline emissions of 122,889 t, so the constraint was not binding and carbon removal could not influence abatement. I tightened the cap to 90% of baseline (110,600 t), at which point it binds and emissions equal the cap exactly.

Baseline emissions with a tight cap: 110,600.25 tonnes CO2
DACCS net emissions: 109,480.25 tonnes CO2
Change in emissions: -13,408.92 tonnes CO2

# Results

Generation rises by exactly the DACCS demand (1680 MWh).
Gross emissions stay at the cap because it binds. 
Net emissions are 109,480 tonnes which is below the cap. 

Interpretation: Since removals sit outside the cap, DACCS is additional to abatement here. The system abates to the cap and the removals come on top. If removals were instead credited within the cap, as in an integrated ETS, gross emissions could rise by up to the removed amount (~1120 tonnes of CO2). So DACCS could substitute for abatement rather than add to it. This is a key policy trade-off raised by potential integration of CDR into the EU ETS (not investigated in this notebook).

Many assumptions were made for this model, including but not limited to:
- No explicit subsidy or financing mechanism for DACCS is modeled here. This represents a hypothetical deployment supported by external policy, for instance EU ETS support. 
- DACCS values were chosen randomly for this exercise, and may or may not represent reality.
- Removals are not credited within the cap.

# Future Directions

- Model DACCS as an extendable link with CO2 accounting, so removals count as negative emissions inside the optimization.
- Compare the cap's marginal abatement cost with DACC's cost per tonne.
- Expand the temporal and spatial scope.
- Consider BECCS.